In [5]:
import numpy as np
import pandas as pd
from pathlib import Path

In [46]:
## merged data from parquet file 

MERGED_PATH = (
    Path.cwd().parent
    / "data"
    / "processed"
    / "merged_data.parquet"
)
merged_df = pd.read_parquet(MERGED_PATH)
print(merged_df["date"].dt.tz)
merged_df.info()


Europe/Istanbul
<class 'pandas.DataFrame'>
RangeIndex: 84696 entries, 0 to 84695
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype                          
---  ------         --------------  -----                          
 0   date           84696 non-null  datetime64[us, Europe/Istanbul]
 1   consumption    84696 non-null  float64                        
 2   year           84696 non-null  int32                          
 3   month          84696 non-null  int32                          
 4   day            84696 non-null  int32                          
 5   weekday        84696 non-null  str                            
 6   national_temp  84696 non-null  float64                        
 7   hour           84696 non-null  int32                          
dtypes: datetime64[us, Europe/Istanbul](1), float64(2), int32(4), str(1)
memory usage: 4.5 MB


In [47]:
merged_df["lag_168"] = merged_df["consumption"].shift(168)
merged_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 84696 entries, 0 to 84695
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype                          
---  ------         --------------  -----                          
 0   date           84696 non-null  datetime64[us, Europe/Istanbul]
 1   consumption    84696 non-null  float64                        
 2   year           84696 non-null  int32                          
 3   month          84696 non-null  int32                          
 4   day            84696 non-null  int32                          
 5   weekday        84696 non-null  str                            
 6   national_temp  84696 non-null  float64                        
 7   hour           84696 non-null  int32                          
 8   lag_168        84528 non-null  float64                        
dtypes: datetime64[us, Europe/Istanbul](1), float64(3), int32(4), str(1)
memory usage: 5.1 MB


In [48]:
merged_df_v2 = merged_df.dropna()
merged_df_v2.info()

<class 'pandas.DataFrame'>
RangeIndex: 84528 entries, 168 to 84695
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype                          
---  ------         --------------  -----                          
 0   date           84528 non-null  datetime64[us, Europe/Istanbul]
 1   consumption    84528 non-null  float64                        
 2   year           84528 non-null  int32                          
 3   month          84528 non-null  int32                          
 4   day            84528 non-null  int32                          
 5   weekday        84528 non-null  str                            
 6   national_temp  84528 non-null  float64                        
 7   hour           84528 non-null  int32                          
 8   lag_168        84528 non-null  float64                        
dtypes: datetime64[us, Europe/Istanbul](1), float64(3), int32(4), str(1)
memory usage: 5.1 MB


In [50]:
#Divide the data train val test-------Baseline model => lag_168
train = merged_df_v2[merged_df_v2["date"] < "2024-09-01"].copy()
val   = merged_df_v2[(merged_df_v2["date"] >= "2024-09-01") & (merged_df_v2["date"] < "2025-09-01")].copy()
test  = merged_df_v2[merged_df_v2["date"] >= "2025-09-01"].copy()

In [53]:
val.info()

<class 'pandas.DataFrame'>
RangeIndex: 8760 entries, 67200 to 75959
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype                          
---  ------         --------------  -----                          
 0   date           8760 non-null   datetime64[us, Europe/Istanbul]
 1   consumption    8760 non-null   float64                        
 2   year           8760 non-null   int32                          
 3   month          8760 non-null   int32                          
 4   day            8760 non-null   int32                          
 5   weekday        8760 non-null   str                            
 6   national_temp  8760 non-null   float64                        
 7   hour           8760 non-null   int32                          
 8   lag_168        8760 non-null   float64                        
dtypes: datetime64[us, Europe/Istanbul](1), float64(3), int32(4), str(1)
memory usage: 541.3 KB


In [ ]:
MAPE = ((val["consumption"]-val["lag_168"]).abs() / val["consumption"]).mean()*100
print("MAPE (%):", MAPE)

MAPE (%): 5.408003353033155


In [59]:
weekday_MAPE = (((val["consumption"]-val["lag_168"]).abs() / val["consumption"])*100).groupby(val["weekday"]).mean()
print("Weekday MAPE (%):", weekday_MAPE)

Weekday MAPE (%): weekday
Friday       5.247795
Monday       6.494273
Saturday     5.442612
Sunday       5.266487
Thursday     4.638080
Tuesday      5.796544
Wednesday    4.972954
dtype: float64


In [60]:
hourly_MAPE = (((val["consumption"]-val["lag_168"]).abs() / val["consumption"])*100).groupby(val["hour"]).mean()
print("Hourly MAPE (%):", hourly_MAPE)

Hourly MAPE (%): hour
0     4.578570
1     4.623891
2     4.701159
3     4.666014
4     4.829724
5     4.964263
6     5.019209
7     5.488737
8     5.995435
9     6.048493
10    5.933750
11    5.973194
12    6.064707
13    6.199935
14    6.379874
15    6.370488
16    6.247113
17    5.781926
18    5.297643
19    5.072760
20    5.000706
21    4.973743
22    4.845067
23    4.735678
dtype: float64
